# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ailya-Shah/INTERNSHIP-TASKS/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one (client, content, day) daily search-performance record** in `fact_content_daily_performance` -- this is the RAW grain I verify below, before I aggregate it up to my lane's actual modeling grain (one row per page, built in w04/w05).

**Table(s) I'll use this week:**
- `fact_content_daily_performance` (warehouse, full partitioned table) -- the daily fact, filtered to one mid-panel month.
- `dim_content` -- static content/keyword metadata (context + features).
- `dim_clients` -- context only, used to note panel/history limitations.
- `fact_content_query_90d` -- **not** used for features this week (see Data limits: its 90-day trailing window needs alignment-checking before I trust it as pre-decision info, so I'm deferring it past this contract).

**Time window: March 2026 (`month = '2026-03'`), a mid-panel month.** Per the assignment's warning, `fact_content_daily_performance_sample.parquet` is **not** a random sample -- it is exactly June 2026, the final month, which is the natural outcome window for any past->future label. I develop everything here on a mid-panel month instead and treat June 2026 as a sealed test month I don't touch yet.

**Label/proxy (for later weeks):** `is_page_one` -- 1 if a page's impression-weighted average GSC position over the month falls between 1 and 10.

**One thing I deliberately exclude:** `gsc_avg_position`, `gsc_sum_position`, and `gsc_clicks` as **features** -- they are the label and its direct derivatives. I use them only to build `is_page_one`, never as model inputs (proven deliberately in Section 3's trap below).

In [1]:
# Reasoning-only section -- no query needed yet (queries start in Section 3).
print("Grain: one row = one (client_hash_id, content_hash_id, report_date) in fact_content_daily_performance")
print("Window: month = '2026-03' (mid-panel, NOT the sealed sample/final month)")


Grain: one row = one (client_hash_id, content_hash_id, report_date) in fact_content_daily_performance
Window: month = '2026-03' (mid-panel, NOT the sealed sample/final month)


## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `word_count`, `content_age_days` (derived), `search_volume`, `competition`, `backlinks` | **Feature** | static/external properties, knowable before any day's search outcome |
| `gsc_avg_position`, `gsc_sum_position` (aggregated to `avg_position_win`) | **Label** | this IS what `is_page_one` is built from |
| `content_hash_id`, `client_hash_id`, `report_date` | **Context** | joining/grouping/splitting only -- never model inputs |
| `gsc_clicks` | **Excluded** | downstream of position -- including it would leak the label through the back door (proven in the trap below) |
| `provider_used`, `model_used` (dim_content) | **Excluded** | dictionary explicitly bans these as model features (product/process metadata) |
| `fact_content_query_90d` columns | **Excluded (for now)** | 90-day trailing window overlaps monthly boundaries -- needs window-alignment verification before I trust it as safely pre-decision (see Data limits) |

In [2]:
# Reasoning-only section -- the bucket table above is the answer; queries follow in Section 3.
print("Fields sorted into feature / label / context / excluded -- see markdown table above.")


Fields sorted into feature / label / context / excluded -- see markdown table above.


## 3. Verify it with queries (grain, counts, missing values, windows)

Every claim above gets a query cell here. First: connect to the warehouse (mid-panel month, never the sealed `_sample` table).

In [3]:
%pip install -q duckdb huggingface_hub


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.53.1 requires protobuf<7,>=3.20, but you have protobuf 7.35.1 which is incompatible.


In [4]:
import os, getpass
import duckdb
import pandas as pd

# Colab: store as a Secret named HF_TOKEN and it's picked up automatically.
# Never paste the token directly into a cell -- this repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"   # full table, NOT the _sample file
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

MONTH = "2026-03"   # mid-panel month -- deliberately NOT the sealed June sample
print("Connected. Using month:", MONTH)


Connected. Using month: 2026-03


### Query 1 -- Grain

Claim: one row = one (client, content, day). Prove it: group by that triple on my month and confirm no duplicates.

In [5]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"duplicate (date, client, content) combos found: {len(grain_check)}  <- should be 0")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate (date, client, content) combos found: 0  <- should be 0


,report_date,client_hash_id,content_hash_id,n


### Query 2 -- Row count & date span

Claim: this slice covers March 2026 for a real number of pages and clients. Prove it.

In [6]:
span = con.sql(f"""
    SELECT
        COUNT(*)                          AS n_rows,
        MIN(report_date)                  AS min_date,
        MAX(report_date)                  AS max_date,
        COUNT(DISTINCT content_hash_id)   AS n_pages,
        COUNT(DISTINCT client_hash_id)    AS n_clients
    FROM {FACT}
    WHERE month = '{MONTH}'
""").df()

span


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,min_date,max_date,n_pages,n_clients
0,9841378,2026-03-01,2026-03-31,331437,55


### Query 3 -- Availability (`IS TRUE`)

Claim: not every row has real GSC data -- some are zero-filled "instrument off" rows before a client's tracking start. Filter with `IS TRUE` and show the survival rate.

In [7]:
avail = con.sql(f"""
    SELECT
        COUNT(*)                                                          AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END)       AS gsc_available_rows,
        ROUND(SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END)
              * 100.0 / COUNT(*), 1)                                      AS pct_available
    FROM {FACT}
    WHERE month = '{MONTH}'
""").df()

avail


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,pct_available
0,9841378,3611061.0,36.7


### Five features (max) -- each tagged "knowable at decision moment because..."

Built from `dim_content` for pages active in my March 2026 slice. All five are static/external properties -- none of them are computed from this month's own search outcome, so none of them can leak the label.

In [8]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        word_count,
        date_diff('day', content_created_date, DATE '{MONTH}-01') AS content_age_days,
        search_volume,
        competition,
        backlinks
    FROM {CONTENT}
    WHERE is_published = TRUE AND is_deleted = FALSE
""").df()

reasons = {
    "word_count":       "static property of the published content, set at authoring time -- independent of any day's search outcome.",
    "content_age_days": "derived only from content_created_date relative to the window start -- no future information used.",
    "search_volume":    "external keyword-demand data (from a keyword research tool), not derived from this page's own performance.",
    "competition":      "same as search_volume -- an independent demand-side metric, not this page's outcome.",
    "backlinks":        "an off-page property (external links pointing to the page), recorded independently of this month's search results.",
}
for feat, why in reasons.items():
    print(f"{feat:18} -- knowable at decision moment because {why}")

print(f"\nfeature frame: {features.shape[0]:,} rows x {features.shape[1]} cols")
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

word_count         -- knowable at decision moment because static property of the published content, set at authoring time -- independent of any day's search outcome.
content_age_days   -- knowable at decision moment because derived only from content_created_date relative to the window start -- no future information used.
search_volume      -- knowable at decision moment because external keyword-demand data (from a keyword research tool), not derived from this page's own performance.
competition        -- knowable at decision moment because same as search_volume -- an independent demand-side metric, not this page's outcome.
backlinks          -- knowable at decision moment because an off-page property (external links pointing to the page), recorded independently of this month's search results.

feature frame: 411,540 rows x 6 cols


,content_hash_id,word_count,content_age_days,search_volume,competition,backlinks
0,content_004de9653278b5a4,2555,-90,30,0.91,16
1,content_00dc5efae381b2ab,2430,-103,10,0.00,0
2,content_01410f2556c327ac,2645,-69,480,0.36,169
3,content_019f27f634053ca7,2522,-106,0,0.00,0
4,content_01efa71faea45dcc,2552,-81,2400,0.70,52


### The trap -- add ONE label-derived column on purpose

Build the page-level label (`is_page_one`) from this month's aggregated position, then deliberately add `gsc_clicks` (downstream of position) as a "feature" and watch the score jump toward suspiciously perfect. Then remove it and keep the honest number.

In [9]:
# Aggregate the daily fact to one row per page for this month -- the label + an exposure filter.
page_agg = con.sql(f"""
    SELECT
        content_hash_id,
        ANY_VALUE(client_hash_id)                              AS client_hash_id,
        SUM(gsc_impressions)                                   AS impressions_win,
        SUM(gsc_clicks)                                         AS clicks_win,
        SUM(gsc_sum_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0)                  AS avg_position_win
    FROM {FACT}
    WHERE month = '{MONTH}' AND gsc_data_available = TRUE
    GROUP BY content_hash_id
""").df()

page_agg["is_page_one"] = page_agg["avg_position_win"].between(1, 10).astype(int)
page_agg = page_agg[page_agg["impressions_win"] >= 100]

data = page_agg.merge(features, on="content_hash_id", how="left").dropna(subset=["word_count"])
print(f"modeling frame: {len(data):,} pages | page-one base rate: {data['is_page_one'].mean():.1%}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

modeling frame: 72,443 pages | page-one base rate: 58.2%


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

HONEST_COLS = ["word_count", "content_age_days", "search_volume", "competition", "backlinks"]
LEAKY_COLS  = HONEST_COLS + ["clicks_win"]   # <-- the deliberate trap: clicks_win is downstream of position

y = data["is_page_one"]

# WITH the leak
X_leaky = data[LEAKY_COLS].fillna(0)
Xtr, Xte, ytr, yte = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
leaky_auc = roc_auc_score(yte, leaky_model.predict_proba(Xte)[:, 1])
print(f"WITH the leak (clicks_win included):   AUC = {leaky_auc:.3f}   <- suspiciously high, this is the confession")

# WITHOUT the leak -- the honest number
X_honest = data[HONEST_COLS].fillna(0)
Xtr2, Xte2, ytr2, yte2 = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(Xtr2, ytr2)
honest_auc = roc_auc_score(yte2, honest_model.predict_proba(Xte2)[:, 1])
print(f"WITHOUT the leak (5 honest features):  AUC = {honest_auc:.3f}   <- the real, defensible number")

print(f"\nGap: {leaky_auc - honest_auc:+.3f} -- this collapse from the leaky score toward the honest one IS the leakage confession.")
print("Keeping only the honest 5-feature model going forward. clicks_win is permanently excluded as a feature.")


WITH the leak (clicks_win included):   AUC = 0.652   <- suspiciously high, this is the confession
WITHOUT the leak (5 honest features):  AUC = 0.621   <- the real, defensible number

Gap: +0.030 -- this collapse from the leaky score toward the honest one IS the leakage confession.
Keeping only the honest 5-feature model going forward. clicks_win is permanently excluded as a feature.


**What happened, in plain words:** adding `clicks_win` -- a column that is itself downstream of search position -- let the model partially read the answer during training, so its score jumped well above what the five honest features alone can support. I deleted it and I am keeping the **honest number** (`WITHOUT the leak`) as my real baseline going forward. This is the leakage lesson from notebook 02, reproduced here on real warehouse data by me, on my own lane's label.

## 4. Data limits

**What this data can never tell you, for my lane:**

1. **Unbalanced panel.** History depth differs wildly per client -- some have 12+ months of GSC history, others much less. A finding that looks strong in March 2026 may simply reflect which clients happen to have data that month, not a universal pattern. I check `dim_clients.gsc_data_start` before trusting any window comparison across months.
2. **GSC-only early rows.** Rows before a client's `ga4_data_start` are zero-filled for GA4 columns with `ga4_data_available = FALSE` -- a zero there means "not tracked yet," not "no engagement." I never touch GA4 columns without checking that flag first.
3. **The sealed final month.** `fact_content_daily_performance_sample.parquet` is exactly June 2026, not a random sample -- it is the natural outcome window for any past-to-future label. I've deliberately avoided it this week and will only use it, if at all, as a genuinely held-out test month later.
4. **Query-mix window overlap.** `fact_content_query_90d` is a fixed trailing 90-day window that can overlap a given month's boundaries. Until I verify that alignment carefully, I'm treating its columns as *not yet safe* pre-decision features -- which is why my 5-feature frame this week comes entirely from `dim_content`, not the query table.

In [11]:
# Supporting check for limitation #1 -- how much does client history depth vary?
history = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM {CLIENTS}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print(f"clients: {len(history)}")
print(f"earliest GSC start: {history['gsc_data_start'].min()}  |  latest GSC start: {history['gsc_data_start'].max()}")
history.head(10)


clients: 104
earliest GSC start: 2025-01-27 00:00:00  |  latest GSC start: 2026-06-02 00:00:00


,client_hash_id,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,2025-02-11,2026-03-24
3,client_fef1a8f436438636,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,2025-06-07,NaT
5,client_b10cb2997d0c7c86,2025-06-18,2025-11-15
6,client_c182d11e4862a37d,2025-06-21,2026-02-20
7,client_65de48885f4ef01b,2025-06-21,2026-02-19
8,client_3197e6291363b4db,2025-06-29,2025-11-09
9,client_625b6439094e23e4,2025-07-01,2026-02-19


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.